# 05 — Staff-facing example alerts

This notebook translates held-out diagnosis predictions into a small set of staff-readable alert cards. The cards demonstrate the proposed reporting format; they are not live BMS alerts. Each example is selected from the chronological held-out evaluation and remains traceable to a saved row-level prediction and sensor values.

In [1]:
from pathlib import Path
import json
import pandas as pd

def find_root():
    for candidate in (Path.cwd(), Path.cwd().parent):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'results').exists():
            return candidate
    raise FileNotFoundError('Open this notebook from the repository or a child directory.')

ROOT = find_root()
RESULTS = ROOT / 'results' / 'diagnosis'
BUILDING_RESULTS = ROOT / 'results' / 'building_specific'
DATA = ROOT / 'data' / 'interim' / 'cleaned' / 'cleaned_ahu.csv'
required = [RESULTS / 'diagnosis_predictions.csv', RESULTS / 'diagnosis_manifest.json', BUILDING_RESULTS / 'building_detection_test_metrics.csv', DATA]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Run the diagnosis pipeline first:\n' + '\n'.join(missing))
manifest = json.loads((RESULTS / 'diagnosis_manifest.json').read_text(encoding='utf-8'))
predictions = pd.read_csv(RESULTS / 'diagnosis_predictions.csv')
test_metrics = pd.read_csv(BUILDING_RESULTS / 'building_detection_test_metrics.csv')
sensor_columns = ['building', 'ahu', 'timestamp', 'label', 'supply_fan', 'supply_air_temperature', 'setpoint_temperature', 'return_temperature', 'valve_position', 'heating_pump_total', 'cooling_pump_total']
sensor_data = pd.read_csv(DATA, usecols=sensor_columns, low_memory=False).reset_index(names='row_id')
print('Evaluation design: chronological')
print('Confidence threshold:', f"{100 * manifest['confidence_threshold']:.0f}%")
print('Rows available for examples:', f"{len(predictions):,}")

Evaluation design: chronological
Confidence threshold: 70%
Rows available for examples: 263,896


## 1. Proposed staff-facing format

The format deliberately uses plain operational language while retaining the model's exact predicted type and confidence. An uncertain diagnosis is shown as `Uncertain` rather than forcing a specific fault name.

In [2]:
chronological = predictions[predictions['design'].eq('chronological')].copy()
chronological['correct'] = chronological['predicted_type'].eq(chronological['true_type'])
chronological = chronological.merge(sensor_data, on=['row_id', 'building', 'ahu', 'timestamp'], how='left', suffixes=('', '_sensor'))

def choose(predicted_type=None, *, uncertain=False, wrong=False):
    candidates = chronological.copy()
    if predicted_type is not None:
        candidates = candidates[candidates['predicted_type'].eq(predicted_type)]
    if uncertain:
        candidates = candidates[candidates['type_abstained']]
    else:
        candidates = candidates[~candidates['type_abstained']]
    if wrong:
        candidates = candidates[~candidates['correct']]
    else:
        candidates = candidates[candidates['correct']]
    return candidates.sort_values('type_confidence', ascending=not uncertain).iloc[0]

examples = [
    ('Correct confident diagnosis', choose('Supply fan fault')),
    ('Uncertain diagnosis', choose(uncertain=True)),
    ('Confident but incorrect diagnosis', choose(wrong=True)),
]

actions = {
    'Supply fan fault': ('Reduced fan signal and abnormal supply-air response.', 'Inspect supply fan operation.'),
    'Heating pump fault': ('Heating-pump signal and heating-temperature response are unusual.', 'Inspect the heating pump and heating-water circulation.'),
    'Cooling pump fault': ('Cooling-pump signal and cooling-temperature response are unusual.', 'Inspect the cooling pump and chilled-water circulation.'),
    'Valve position fault': ('Valve position and air-temperature response are inconsistent.', 'Inspect valve operation and the associated temperature response.'),
    'Return air temperature fault': ('Return-air temperature differs from the expected operating relationship.', 'Inspect return-air sensing and the affected air path.'),
    'Supply air temperature fault': ('Supply-air temperature differs from the expected setpoint relationship.', 'Inspect supply-air sensing and the heating/cooling response.'),
}

cards = []
for example_name, row in examples:
    predicted = 'Uncertain' if bool(row['type_abstained']) else str(row['predicted_type'])
    evidence, action = actions.get(predicted, ('The sensor relationships are unusual.', 'Inspect the affected AHU and review the supporting trends.'))
    cards.append({
        'Example': example_name,
        'Status': 'Fault detected',
        'Building': str(row['building']).title(),
        'AHU': str(row['ahu']),
        'Timestamp': str(row['timestamp']),
        'Likely fault type': predicted,
        'Confidence': f"{100 * float(row['type_confidence']):.1f}%",
        'Evidence': evidence,
        'Recommended action': action,
        'Audit outcome': 'Correct' if bool(row['correct']) else 'Incorrect/uncertain example',
    })
cards = pd.DataFrame(cards)
display(cards.drop(columns=['Audit outcome']))

,Example,Status,Building,AHU,Timestamp,Likely fault type,Confidence,Evidence,Recommended action
0,Correct confident diagnosis,Fault detected,Hospital,AHU-N02,2023-12-02 08:00:00,Supply fan fault,92.0%,Reduced fan signal and abnormal supply-air res...,Inspect supply fan operation.
1,Uncertain diagnosis,Fault detected,Auditorium,AHU-04,2022-12-27 02:00:00,Uncertain,69.8%,The sensor relationships are unusual.,Inspect the affected AHU and review the suppor...
2,Confident but incorrect diagnosis,Fault detected,Auditorium,AHU-10,2022-12-23 07:00:00,Return air temperature fault,70.2%,Return-air temperature differs from the expect...,Inspect return-air sensing and the affected ai...


## 2. Model reliability labels

These labels provide a compact interpretation of held-out detection performance. They summarise the historical evaluation and are not a replacement for the row-level confidence threshold or for commissioning the system on live building data.

- **Strong on this evaluation** — F1 score of at least 0.90.
- **Promising; calibrate before use** — F1 score from 0.70 to 0.89.
- **Not reliable for deployment** — F1 score below 0.70.

In [3]:
def reliability_label(f1):
    if f1 >= 0.90:
        return 'Strong on this evaluation'
    if f1 >= 0.70:
        return 'Promising; calibrate before use'
    return 'Not reliable for deployment'

reliability = test_metrics[test_metrics['model'].eq('xgboost')][
    ['building', 'design', 'f1', 'balanced_accuracy', 'false_positive_rate', 'coverage']
].copy()
reliability['qualitative assessment'] = reliability['f1'].map(reliability_label)
display(reliability.sort_values(['building', 'design']).round(3))

,building,design,f1,balanced_accuracy,false_positive_rate,coverage,qualitative assessment
0,auditorium,building_ahu_held_out,0.997,0.997,0.002,1.0,Strong on this evaluation
1,auditorium,building_chronological,1.000,1.000,0.000,1.0,Strong on this evaluation
2,hospital,building_ahu_held_out,0.077,0.507,0.028,1.0,Not reliable for deployment
3,hospital,building_chronological,0.880,0.958,0.043,1.0,Promising; calibrate before use
4,office,building_ahu_held_out,0.997,0.997,0.000,1.0,Strong on this evaluation
5,office,building_chronological,0.996,0.997,0.001,1.0,Strong on this evaluation


## 3. Why these examples are useful

The three examples demonstrate the important behaviours of the proposed reporting layer:

- a confident and correct diagnosis can name a likely fault;
- a low-confidence result is reported as `Uncertain`;
- a high-confidence error is possible, so the output supports inspection rather than automatic maintenance.

The examples come from labelled historical test rows. They demonstrate the communication format, not the exact onset or duration of a physical fault.

## 4. Audit trail for examples

The table below records whether each displayed historical example matched its labelled fault type.

In [4]:
display(cards[['Example', 'Audit outcome', 'Likely fault type', 'Confidence']])

,Example,Audit outcome,Likely fault type,Confidence
0,Correct confident diagnosis,Correct,Supply fan fault,92.0%
1,Uncertain diagnosis,Correct,Uncertain,69.8%
2,Confident but incorrect diagnosis,Incorrect/uncertain example,Return air temperature fault,70.2%
